# 🩺 Medical Reasoning LLM — QLoRA Training
## Fine-tuning Qwen2.5-3B-Instruct on Medical Chain-of-Thought QA

| Config | Value |
|---|---|
| Base model | Qwen/Qwen2.5-3B-Instruct |
| Dataset | FreedomIntelligence/medical-o1-reasoning-SFT |
| Method | QLoRA (4-bit NF4, LoRA r=16) |
| Training samples | 5,000 |
| Hardware | T4 GPU (Google Colab) |
| Est. time | ~4 hours |

### Before you start
1. **Runtime → Change runtime type → T4 GPU**
2. Run cells top to bottom
3. Each cell is independent and labelled

> ⚠️ This model is a research prototype — not for clinical use.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 🔧 Cell 1 — Environment Check

In [ ]:
import subprocess, sys, os

# ── GPU check ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)

try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
         "--format=csv,noheader"],
        stderr=subprocess.DEVNULL
    ).decode().strip()
    print(f"  GPU      : {gpu_info}")
except Exception:
    print("  ⚠️  No GPU detected. Change runtime to T4 GPU.")
    print("  Runtime → Change runtime type → GPU → T4")

print(f"  Python   : {sys.version.split()[0]}")

# ── Disk space ────────────────────────────────────────────────────────────────
try:
    disk = subprocess.check_output(["df", "-h", "/"]).decode().split("\n")[1].split()
    print(f"  Disk     : {disk[3]} free / {disk[1]} total")
except Exception:
    pass

# ── Colab detection ───────────────────────────────────────────────────────────
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print(f"  Colab    : {IS_COLAB}")
print("=" * 60)

ENVIRONMENT CHECK
  GPU      : Tesla T4, 15360 MiB, 14913 MiB
  Python   : 3.12.13
  Disk     : 66G free / 113G total
  Colab    : True


## 📦 Cell 2 — Install Dependencies
*(~3 minutes on first run)*

In [ ]:
%%capture install_output
!pip install \
    torch \
    transformers==4.43.4 \
    tokenizers==0.19.1 \
    accelerate==0.30.1 \
    datasets==2.19.2 \
    peft==0.11.1 \
    trl==0.9.4 \
    bitsandbytes>=0.44.0 \
    rouge-score==0.1.2 \
    evaluate==0.4.2 \
    safetensors==0.4.3 \
    huggingface_hub==0.23.4 \
    tqdm rich pyyaml python-dotenv -q

In [ ]:
import transformers, peft, trl, bitsandbytes, datasets
print("✓ All packages installed")
print(f"  transformers : {transformers.__version__}")
print(f"  peft         : {peft.__version__}")
print(f"  trl          : {trl.__version__}")
print(f"  bitsandbytes : {bitsandbytes.__version__}")
print(f"  datasets     : {datasets.__version__}")

KeyboardInterrupt: 

## 💾 Cell 3 — Mount Google Drive *(Optional but Recommended)*
Saves checkpoints to Drive so you don't lose them if the Colab session disconnects.

In [ ]:
MOUNT_DRIVE = True   # ← Set to False to skip

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT_DIR = "/content/drive/MyDrive/medical-reasoning-llm/results"
        print(f"✓ Drive mounted. Outputs → {OUTPUT_DIR}")
    except Exception as e:
        print(f"Drive mount failed: {e}")
        MOUNT_DRIVE = False

if not MOUNT_DRIVE:
    OUTPUT_DIR = "/content/results"
    print(f"Using local Colab storage: {OUTPUT_DIR}")
    print("⚠️  Session data will be lost when Colab disconnects.")

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "final_adapter")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Drive mounted. Outputs → /content/drive/MyDrive/medical-reasoning-llm/results


## ⚙️ Cell 4 — Configuration
All hyperparameters in one place. Edit here if needed.

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME    = "Qwen/Qwen2.5-3B-Instruct"
HF_TOKEN      = ""   # Optional: set if needed for private models
                     # Or: from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')

# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET_NAME     = "FreedomIntelligence/medical-o1-reasoning-SFT"
NUM_SAMPLES      = 5_000   # Use 25_000 for full dataset (needs ~16h on T4)
TRAIN_RATIO      = 0.80
VAL_RATIO        = 0.10
TEST_RATIO       = 0.10
MAX_SEQ_LENGTH   = 2048
SEED             = 42

# ── QLoRA ─────────────────────────────────────────────────────────────────────
LORA_R           = 16
LORA_ALPHA       = 32      # Always 2× r
LORA_DROPOUT     = 0.05
TARGET_MODULES   = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"]

# ── Training ──────────────────────────────────────────────────────────────────
NUM_EPOCHS       = 3
BATCH_SIZE       = 2        # Per-device; T4 max with seq_len=2048
GRAD_ACCUM       = 4        # Effective batch = BATCH_SIZE × GRAD_ACCUM = 8
LEARNING_RATE    = 2e-4
LR_SCHEDULER     = "cosine"
WARMUP_RATIO     = 0.05
WEIGHT_DECAY     = 0.01
LOGGING_STEPS    = 25
EVAL_STEPS       = 100
SAVE_STEPS       = 200
FP16             = True

EFFECTIVE_BATCH  = BATCH_SIZE * GRAD_ACCUM
APPROX_STEPS     = (int(NUM_SAMPLES * TRAIN_RATIO) // EFFECTIVE_BATCH) * NUM_EPOCHS

print("=" * 55)
print("  TRAINING CONFIGURATION")
print("=" * 55)
print(f"  Model         : {MODEL_NAME}")
print(f"  Dataset       : {DATASET_NAME}")
print(f"  Samples       : {NUM_SAMPLES:,}")
print(f"  LoRA r/alpha  : {LORA_R} / {LORA_ALPHA}  (scaling={LORA_ALPHA/LORA_R})")
print(f"  Epochs        : {NUM_EPOCHS}")
print(f"  Effective batch: {EFFECTIVE_BATCH}")
print(f"  Learning rate  : {LEARNING_RATE:.1e}")
print(f"  Approx steps   : ~{APPROX_STEPS:,}")
print(f"  Max seq length : {MAX_SEQ_LENGTH}")
print(f"  Output dir     : {OUTPUT_DIR}")
print("=" * 55)

  TRAINING CONFIGURATION
  Model         : Qwen/Qwen2.5-3B-Instruct
  Dataset       : FreedomIntelligence/medical-o1-reasoning-SFT
  Samples       : 5,000
  LoRA r/alpha  : 16 / 32  (scaling=2.0)
  Epochs        : 3
  Effective batch: 8
  Learning rate  : 2.0e-04
  Approx steps   : ~1,500
  Max seq length : 2048
  Output dir     : /content/drive/MyDrive/medical-reasoning-llm/results


##  Cell 5 — Load & Split Dataset

In [ ]:
import random
from datasets import load_dataset, DatasetDict

print(f"Loading {DATASET_NAME}...")
full_ds = load_dataset(DATASET_NAME,"en", split="train")
print(f"Full dataset: {len(full_ds):,} samples")

# Draw reproducible subset
random.seed(SEED)
indices = sorted(random.sample(range(len(full_ds)), NUM_SAMPLES))
subset = full_ds.select(indices)

# Split
shuffled = subset.shuffle(seed=SEED)
n = len(shuffled)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)
n_test  = n - n_train - n_val

splits = DatasetDict({
    "train"      : shuffled.select(range(0, n_train)),
    "validation" : shuffled.select(range(n_train, n_train + n_val)),
    "test"       : shuffled.select(range(n_train + n_val, n)),
})

print(f"\nSplits → train: {len(splits['train']):,} | "
      f"val: {len(splits['validation']):,} | test: {len(splits['test']):,}")

# Quick sanity check
sample = splits["train"][0]
print(f"\nSample question (first 120 chars):")
print(f"  {sample['Question'][:120]}...")

Loading FreedomIntelligence/medical-o1-reasoning-SFT...


Full dataset: 19,704 samples

Splits → train: 4,000 | val: 500 | test: 500

Sample question (first 120 chars):
  Which diseases predispose a patient to develop osteosarcoma?...


##  Cell 6 — Load Tokenizer & Format Dataset

In [ ]:
from transformers import AutoTokenizer

print(f"Loading tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Tokenizer loaded")
print(f"  vocab_size : {tokenizer.vocab_size:,}")
print(f"  pad_token  : {tokenizer.pad_token!r}")
print(f"  eos_token  : {tokenizer.eos_token!r}")

# ── System message ────────────────────────────────────────────────────────────
SYSTEM_MESSAGE = (
    "You are an expert physician with deep knowledge of internal medicine, "
    "cardiology, neurology, and emergency medicine. When presented with a "
    "clinical scenario, you reason through it systematically before providing "
    "your final answer. Your reasoning should include: symptom analysis, "
    "relevant anatomy and physiology, differential diagnosis with reasoning, "
    "and evidence-based management. Always separate your thinking process "
    "from your final answer."
)

# ── Format function ───────────────────────────────────────────────────────────
def format_example(row):
    assistant_content = (
        "Let me reason through this step by step:\n"
        f"{row['Complex_CoT']}\n\n"
        "Final Answer:\n"
        f"{row['Response']}"
    )
    messages = [
        {"role": "system",    "content": SYSTEM_MESSAGE},
        {"role": "user",      "content": row["Question"]},
        {"role": "assistant", "content": assistant_content},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    }

# ── Apply to train and val splits ────────────────────────────────────────────
print("\nApplying chat template...")
formatted = {}
for split_name in ["train", "validation"]:
    ds = splits[split_name].map(format_example, num_proc=2, desc=f"Formatting {split_name}")
    before = len(ds)
    ds = ds.filter(
        lambda x: len(tokenizer.encode(x["text"], add_special_tokens=False)) <= MAX_SEQ_LENGTH,
        desc="Filtering long sequences"
    )
    print(f"  {split_name}: {before} → {len(ds)} (filtered {before - len(ds)} long sequences)")
    formatted[split_name] = ds

# Preview one formatted example
sample_text = formatted["train"][0]["text"]
sample_tokens = len(tokenizer.encode(sample_text))
print(f"\nFormatted example ({sample_tokens} tokens):")
print(sample_text[:600])
print("..." if len(sample_text) > 600 else "")

Loading tokenizer: Qwen/Qwen2.5-3B-Instruct...
✓ Tokenizer loaded
  vocab_size : 151,643
  pad_token  : '<|endoftext|>'
  eos_token  : '<|im_end|>'

Applying chat template...
  train: 4000 → 4000 (filtered 0 long sequences)
  validation: 500 → 500 (filtered 0 long sequences)

Formatted example (762 tokens):
<|im_start|>system
You are an expert physician with deep knowledge of internal medicine, cardiology, neurology, and emergency medicine. When presented with a clinical scenario, you reason through it systematically before providing your final answer. Your reasoning should include: symptom analysis, relevant anatomy and physiology, differential diagnosis with reasoning, and evidence-based management. Always separate your thinking process from your final answer.<|im_end|>
<|im_start|>user
Which diseases predispose a patient to develop osteosarcoma?<|im_end|>
<|im_start|>assistant
Let me reason th
...


##  Cell 7 — Load Base Model (4-bit NF4)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

print("Loading base model with 4-bit NF4 quantization...")
print("(This downloads ~6 GB on first run, then uses the cache)\n")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,    # nested quantization: extra 0.4 bits/param saved
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache       = False   # disable KV cache during training
model.config.pretraining_tp  = 1       # needed for gradient flow with QLoRA

# Memory report
total_params     = sum(p.numel() for p in model.parameters())
mem_allocated_gb = torch.cuda.memory_allocated() / 1024**3

print(f"✓ Model loaded")
print(f"  Parameters     : {total_params/1e9:.2f}B")
print(f"  GPU memory used: {mem_allocated_gb:.2f} GB")
print(f"  Device map     : {model.hf_device_map}")

Loading base model with 4-bit NF4 quantization...
(This downloads ~6 GB on first run, then uses the cache)



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✓ Model loaded
  Parameters     : 1.70B
  GPU memory used: 5.89 GB
  Device map     : {'': 0}


##  Cell 8 — Attach LoRA Adapters

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

# Step 1: Prepare frozen quantized base for gradient flow
model = prepare_model_for_kbit_training(model)

# Step 2: Enable gradient checkpointing (trades compute for VRAM)
model.enable_input_require_grads()

# Step 3: Inject LoRA adapter layers
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=TARGET_MODULES,
)
model = get_peft_model(model, lora_config)

# Parameter summary
total      = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen     = total - trainable

print("=" * 55)
print("  PARAMETER SUMMARY")
print("=" * 55)
print(f"  Trainable (LoRA adapters) : {trainable:>12,}  ({100*trainable/total:.4f}%)")
print(f"  Frozen    (quantized base): {frozen:>12,}  ({100*frozen/total:.4f}%)")
print(f"  Total                     : {total:>12,}")
print("=" * 55)
print(f"  LoRA r={LORA_R}, alpha={LORA_ALPHA}, scaling={LORA_ALPHA/LORA_R:.1f}")
print(f"  Targets: {TARGET_MODULES}")

  PARAMETER SUMMARY
  Trainable (LoRA adapters) :   29,933,568  (1.7317%)
  Frozen    (quantized base): 1,698,672,640  (98.2683%)
  Total                     : 1,728,606,208
  LoRA r=16, alpha=32, scaling=2.0
  Targets: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


##  Cell 9 — Training Callbacks

In [ ]:
import time
from transformers import TrainerCallback, TrainerControl, TrainerState, TrainingArguments

# ── Fixed clinical question used to monitor training quality ─────────────────
MONITOR_QUESTION = (
    "A 58-year-old male with a 20-pack-year smoking history presents with "
    "3 weeks of worsening exertional dyspnea, orthopnea, and bilateral leg edema. "
    "JVP is elevated at 10 cm. Chest X-ray shows cardiomegaly and bilateral pleural "
    "effusions. BNP is 1,450 pg/mL. Echo shows EF 25%. What is the diagnosis and "
    "initial management?"
)

class SampleGenerationCallback(TrainerCallback):
    """Generates a clinical example at each eval to watch reasoning improve."""

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        step = state.global_step
        print(f"\n{'='*65}")
        print(f"  SAMPLE GENERATION — Step {step}")
        print(f"{'='*65}")

        messages = [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user",   "content": MONITOR_QUESTION},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt",
                           add_special_tokens=False).to(model.device)

        model.eval()
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=400,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        model.train()

        new_toks = out_ids[0][inputs["input_ids"].shape[1]:]
        output   = tokenizer.decode(new_toks, skip_special_tokens=True)

        print(f"Q: {MONITOR_QUESTION[:100]}...")
        print(f"{'─'*65}")
        print(output[:800])
        print()

        # Log to file
        with open(f"{OUTPUT_DIR}/sample_generations.txt", "a") as f:
            f.write(f"\n{'='*65}\nSTEP {step} | {time.strftime('%H:%M:%S')}\n")
            f.write(f"Q: {MONITOR_QUESTION}\n\nA:\n{output}\n")

print("✓ SampleGenerationCallback defined")

✓ SampleGenerationCallback defined


##  Cell 10 — Setup SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig

CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints"

sft_config = SFTConfig(
    # ── Paths ──────────────────────────────────────────────────────────────
    output_dir            = CHECKPOINT_DIR,

    # ── Core training ──────────────────────────────────────────────────────
    num_train_epochs      = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate         = LEARNING_RATE,
    lr_scheduler_type     = LR_SCHEDULER,
    warmup_ratio          = WARMUP_RATIO,
    weight_decay          = WEIGHT_DECAY,
    max_grad_norm         = 1.0,

    # ── Precision ──────────────────────────────────────────────────────────
    fp16                  = FP16,
    bf16                  = False,     # set True for A100

    # ── Optimizer ──────────────────────────────────────────────────────────
    optim                 = "paged_adamw_32bit",   # memory-efficient

    # ── Logging & checkpoints ──────────────────────────────────────────────
    logging_steps         = LOGGING_STEPS,
    eval_strategy         = "steps",
    eval_steps            = EVAL_STEPS,
    save_strategy         = "steps",
    save_steps            = SAVE_STEPS,
    save_total_limit      = 3,
    load_best_model_at_end= True,
    metric_for_best_model = "eval_loss",
    greater_is_better     = False,

    # ── SFT-specific ───────────────────────────────────────────────────────
    max_seq_length        = MAX_SEQ_LENGTH,
    dataset_text_field    = "text",
    packing               = False,

    # ── Misc ───────────────────────────────────────────────────────────────
    remove_unused_columns = True,
    report_to             = "none",   # change to "wandb" if tracking
    dataloader_num_workers= 2,
)

trainer = SFTTrainer(
    model        = model,
    tokenizer    = tokenizer,
    train_dataset= formatted["train"],
    eval_dataset = formatted["validation"],
    args         = sft_config,
    callbacks    = [SampleGenerationCallback()],
)

train_count = len(formatted["train"])
val_count   = len(formatted["validation"])
steps_per_epoch = train_count // EFFECTIVE_BATCH
total_steps     = steps_per_epoch * NUM_EPOCHS

print("=" * 55)
print("  TRAINER READY")
print("=" * 55)
print(f"  Train samples   : {train_count:,}")
print(f"  Val   samples   : {val_count:,}")
print(f"  Steps / epoch   : ~{steps_per_epoch:,}")
print(f"  Total steps     : ~{total_steps:,}")
print(f"  Eval  every     : {EVAL_STEPS} steps")
print(f"  Save  every     : {SAVE_STEPS} steps")
print(f"  Checkpoint dir  : {CHECKPOINT_DIR}")
print("=" * 55)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

  TRAINER READY
  Train samples   : 4,000
  Val   samples   : 500
  Steps / epoch   : ~500
  Total steps     : ~1,500
  Eval  every     : 100 steps
  Save  every     : 200 steps
  Checkpoint dir  : /content/drive/MyDrive/medical-reasoning-llm/results/checkpoints


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


##  Cell 11 — TRAIN
⏱️ *Estimated time: ~4 hours on T4*

Watch loss drop and the qualitative sample improve every 100 steps.

In [ ]:
# ── Permanent fix for PyTorch 2.6 checkpoint loading ─────────────────────────
import torch

_orig_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs['weights_only'] = False   # trust our own checkpoints
    return _orig_load(*args, **kwargs)

torch.load = _patched_load
print("✓ torch.load patched — checkpoint resume will work now")

✓ torch.load patched — checkpoint resume will work now


In [ ]:
print("Starting training...")
print("  Tip: Every 100 steps a clinical scenario is generated")
print("  so you can watch reasoning quality improve live.\n")

# Fix for PyTorch 2.6 checkpoint loading
import numpy as np
torch.serialization.add_safe_globals([np._core.multiarray._reconstruct])

start_time = time.time()
train_result = trainer.train(resume_from_checkpoint=True)
elapsed_min  = (time.time() - start_time) / 60

print("\n" + "=" * 55)
print("  TRAINING COMPLETE")
print("=" * 55)
print(f"  Train loss    : {train_result.training_loss:.4f}")
print(f"  Total steps   : {train_result.global_step}")
print(f"  Runtime (min) : {elapsed_min:.1f}")
print(f"  Runtime (hr)  : {elapsed_min/60:.2f}")
print("=" * 55)

Starting training...
  Tip: Every 100 steps a clinical scenario is generated
  so you can watch reasoning quality improve live.



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
700,1.187400,1.281485
800,1.201600,1.277807
900,1.193700,1.274782
1000,1.181300,1.272087



  SAMPLE GENERATION — Step 700


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_

...

Q: A 58-year-old male with a 20-pack-year smoking history presents with 3 weeks of worsening exertional...
─────────────────────────────────────────────────────────────────
Let me reason through this step by step:
Alright, let's see what we've got here. We have a 58-year-old guy who has been smoking for quite some time—20 pack-years to be exact. That’s definitely concerning for heart issues.

He's been having trouble breathing more than usual over the past few weeks. It sounds like he's struggling both when he exercises and even while lying down. Hmm, that suggests something serious might be going on in his heart.

Now, there's also swelling in his legs and he's feeling really tired. Oh, and I can't forget about the jugular venous pressure being high at 10 cm. This could mean fluid is building up somewhere important.

The chest X-ray shows his heart looks bigger than normal, plus there are fluid pockets around both lungs. Not good signs at all.

His BNP level


  SAMPLE GENERATION — St

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



  SAMPLE GENERATION — Step 1500


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_

Q: A 58-year-old male with a 20-pack-year smoking history presents with 3 weeks of worsening exertional...
─────────────────────────────────────────────────────────────────
Let me reason through this step by step:
Alright, let's think about what's going on here. We've got a 58-year-old guy who has been smoking for quite some time—20 pack-years—and now he's having trouble breathing when he exerts himself. That sounds pretty serious.

He’s also experiencing some other symptoms like feeling short of breath while lying down and swelling in his legs. Hmm, that definitely suggests something is up with his heart or lungs.

Now, looking at his chest X-ray, we see there's more to this story. The image shows his heart is enlarged, which isn't good news, but wait, there are also fluid collections around both lungs. This makes me think about fluid building up because of pressure issues somewhere.

Oh, and the jugular venous pressure being high at 10 cm—it points towards 


  TRAINING COMPLETE
  Tr

##  Cell 12 — Save Adapter

In [ ]:
import os

# Save LoRA adapter only (~100 MB, not the full 6 GB model)
print(f"Saving adapter to: {ADAPTER_DIR}")
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Report saved files
print("\nSaved files:")
for f in sorted(os.listdir(ADAPTER_DIR)):
    size_mb = os.path.getsize(f"{ADAPTER_DIR}/{f}") / 1024**2
    print(f"  {f:<45} {size_mb:>7.2f} MB")

# ── Optional: push to HuggingFace Hub ────────────────────────────────────────
PUSH_TO_HUB  = False   # ← Set True after training to share publicly
HF_REPO_ID   = "YOUR_USERNAME/medical-reasoning-qwen2.5-3b-qlora"

if PUSH_TO_HUB:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub(HF_REPO_ID)
    tokenizer.push_to_hub(HF_REPO_ID)
    print(f"\n✓ Pushed to HuggingFace Hub: {HF_REPO_ID}")

print("\n✓ Adapter saved.")
print(f"  Reload anytime with:")
print(f"  from peft import PeftModel")
print(f"  model = PeftModel.from_pretrained(base_model, '{ADAPTER_DIR}')")

Saving adapter to: /content/drive/MyDrive/medical-reasoning-llm/results/final_adapter

Saved files:
  README.md                                        0.00 MB
  adapter_config.json                              0.00 MB
  adapter_model.safetensors                      114.25 MB
  added_tokens.json                                0.00 MB
  merges.txt                                       1.59 MB
  special_tokens_map.json                          0.00 MB
  tokenizer.json                                   6.71 MB
  tokenizer_config.json                            0.01 MB
  training_args.bin                                0.01 MB
  vocab.json                                       2.65 MB

✓ Adapter saved.
  Reload anytime with:
  from peft import PeftModel
  model = PeftModel.from_pretrained(base_model, '/content/drive/MyDrive/medical-reasoning-llm/results/final_adapter')


##  Cell 13 — Quick Evaluation on Test Set
*Computes accuracy and ROUGE-L on a 50-sample mini-test.*

In [ ]:
from rouge_score import rouge_scorer as rouge_lib

def extract_answer(text):
    """Parse final answer from generated output."""
    if "Final Answer:" in text:
        return text.split("Final Answer:", 1)[-1].strip()
    return text.strip().split("\n\n")[-1].strip()

def extract_reasoning(text):
    """Parse reasoning chain from generated output."""
    if "Let me reason" in text and "Final Answer:" in text:
        chain = text.split("Let me reason", 1)[-1]
        chain = chain.split("Final Answer:", 1)[0]
        return chain.strip()
    return text.strip()

print("Running quick eval on 50 test examples...")
test_sample = splits["test"].select(range(50))

scorer = rouge_lib.RougeScorer(["rougeL"], use_stemmer=False)

pred_answers, gold_answers, pred_chains, gold_chains = [], [], [], []

model.eval()
for row in test_sample:
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": row["Question"]},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_toks = out_ids[0][inputs["input_ids"].shape[1]:]
    output = tokenizer.decode(new_toks, skip_special_tokens=True)

    pred_answers.append(extract_answer(output).lower().strip())
    gold_answers.append(row["Response"].lower().strip())
    pred_chains.append(extract_reasoning(output))
    gold_chains.append(row["Complex_CoT"])

# Accuracy (exact match, normalized)
n = len(pred_answers)
correct = sum(p == g for p, g in zip(pred_answers, gold_answers))
accuracy = correct / n

# ROUGE-L
rouge_scores = [
    scorer.score(g, p)["rougeL"].fmeasure
    for p, g in zip(pred_chains, gold_chains)
]
avg_rouge_l = sum(rouge_scores) / len(rouge_scores)

print("\n" + "=" * 55)
print("  QUICK EVAL RESULTS  (50 test examples)")
print("=" * 55)
print(f"  Answer accuracy  : {accuracy:.4f}  ({correct}/{n})")
print(f"  ROUGE-L (CoT)    : {avg_rouge_l:.4f}")
print("=" * 55)
print("\nRun full eval with: python scripts/evaluate.py --compare_base")

Running quick eval on 50 test examples...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_


  QUICK EVAL RESULTS  (50 test examples)
  Answer accuracy  : 0.0000  (0/50)
  ROUGE-L (CoT)    : 0.1972

Run full eval with: python scripts/evaluate.py --compare_base


##  Cell 14 — Inference Demo
Ask the trained model any clinical question.

In [ ]:
DEMO_QUESTIONS = [
    (
        "A 28-year-old woman presents with 3 days of progressive severe headache, "
        "photophobia, neck stiffness and fever of 39.2°C. She has no rash. "
        "What is the most likely diagnosis and immediate management?"
    ),
    (
        "A 72-year-old male with known COPD and 50 pack-year history presents "
        "with unintentional weight loss of 8 kg over 3 months, hemoptysis, "
        "and a new right upper lobe mass on CT chest. What is the most likely "
        "diagnosis and what investigations would you order?"
    ),
    (
        "A 45-year-old woman presents with fatigue, cold intolerance, weight gain, "
        "constipation, and dry skin for 6 months. Her TSH is 18 mIU/L, "
        "free T4 is 0.4 ng/dL. What is the diagnosis and treatment?"
    ),
]

def ask_model(question, max_new_tokens=600):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       add_special_tokens=False).to(model.device)
    model.eval()
    t0 = time.time()
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - t0
    new_toks = out_ids[0][inputs["input_ids"].shape[1]:]
    output   = tokenizer.decode(new_toks, skip_special_tokens=True)
    tokens   = len(new_toks)
    return output, tokens, elapsed

print("=" * 70)
print("  MEDICAL REASONING LLM — INFERENCE DEMO")
print("=" * 70)

for i, question in enumerate(DEMO_QUESTIONS, 1):
    output, n_tokens, t = ask_model(question)
    reasoning = extract_reasoning(output)
    answer    = extract_answer(output)

    print(f"\n{'─'*70}")
    print(f"  QUESTION {i}  ({n_tokens} tokens | {t:.1f}s)")
    print(f"{'─'*70}")
    print(f"Q: {question[:130]}...")
    print(f"\n[REASONING CHAIN]")
    print(reasoning[:800])
    print(f"\n[FINAL ANSWER]")
    print(answer[:400])

print(f"\n{'─'*70}")
print("⚠️  RESEARCH PROTOTYPE — Not for clinical use.")
print(f"{'─'*70}")

  MEDICAL REASONING LLM — INFERENCE DEMO

──────────────────────────────────────────────────────────────────────
  QUESTION 1  (546 tokens | 99.7s)
──────────────────────────────────────────────────────────────────────
Q: A 28-year-old woman presents with 3 days of progressive severe headache, photophobia, neck stiffness and fever of 39.2°C. She has ...

[REASONING CHAIN]
through this step by step:
Alright, let's think about what's going on here. We've got a young woman who's been dealing with some pretty intense headaches for three whole days. That sounds rough. And she's also experiencing photophobia—basically, she can't stand bright lights—and her neck feels really stiff. Plus, there's a fever that's climbing up to 39.2 degrees Celsius.

Now, when I see these symptoms together, especially the combination of headache, neck stiffness, and fever, my mind immediately jumps to something infectious or inflammatory in nature. Oh, and don't forget the photophobia; that often points towards 

In [ ]:
#For short evaluation we will run cell 2,3,4, and below single reload cell (replaces Cell 5-10)
# ── Fast reload from saved adapter ───────────────────────────────────────────
import torch, random, time
from datasets import load_dataset, DatasetDict
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from rouge_score import rouge_scorer as rouge_lib
import re

# Config (must match what you trained with)
MODEL_NAME   = "Qwen/Qwen2.5-3B-Instruct"
DATASET_NAME = "FreedomIntelligence/medical-o1-reasoning-SFT"
ADAPTER_DIR  = "/content/drive/MyDrive/medical-reasoning-llm/results/final_adapter"
NUM_SAMPLES  = 5_000
SEED         = 42
SYSTEM_MESSAGE = (
    "You are an expert physician with deep knowledge of internal medicine, "
    "cardiology, neurology, and emergency medicine. When presented with a "
    "clinical scenario, you reason through it systematically before providing "
    "your final answer. Always separate your thinking process from your final answer."
)

# ── Load test split ───────────────────────────────────────────────────────────
print("Loading test split...")
random.seed(SEED)
full_ds = load_dataset(DATASET_NAME, "en", split="train")
indices = sorted(random.sample(range(len(full_ds)), NUM_SAMPLES))
subset  = full_ds.select(indices).shuffle(seed=SEED)
n       = len(subset)
n_train, n_val = int(n*0.80), int(n*0.10)
test_ds = subset.select(range(n_train + n_val, n))
test_sample = test_ds.select(range(50))
print(f"Test split: {len(test_ds)} | Using 50 samples for eval")

# ── Load tokenizer ────────────────────────────────────────────────────────────
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, padding_side="left"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Load model + adapter ──────────────────────────────────────────────────────
print("Loading base model (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True,
)
base_model.config.use_cache = True

print(f"Loading adapter from: {ADAPTER_DIR}")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
model.eval()
print("✓ Model + adapter ready")

Loading test split...


Test split: 500 | Using 50 samples for eval
Loading tokenizer...
Loading base model (4-bit)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loading adapter from: /content/drive/MyDrive/medical-reasoning-llm/results/final_adapter
✓ Model + adapter ready


####### Trying out for short evaluation

In [ ]:
#For short evaluation we will run cell 2,3,4, and below single reload cell (replaces Cell 5-10)
# ── Fast reload from saved adapter ───────────────────────────────────────────
import torch, random
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig
from peft import PeftModel

MODEL_NAME   = "Qwen/Qwen2.5-3B-Instruct"
DATASET_NAME = "FreedomIntelligence/medical-o1-reasoning-SFT"
ADAPTER_DIR  = "/content/drive/MyDrive/medical-reasoning-llm/results/final_adapter"
NUM_SAMPLES  = 5_000
SEED         = 42
SYSTEM_MESSAGE = (
    "You are an expert physician with deep knowledge of internal medicine, "
    "cardiology, neurology, and emergency medicine. When presented with a "
    "clinical scenario, you reason through it systematically before providing "
    "your final answer. Always separate your thinking process from your final answer."
)

# ── Load test split ───────────────────────────────────────────────────────────
print("Loading test split...")
random.seed(SEED)
full_ds = load_dataset(DATASET_NAME, "en", split="train")
indices = sorted(random.sample(range(len(full_ds)), NUM_SAMPLES))
subset  = full_ds.select(indices).shuffle(seed=SEED)
n       = len(subset)
n_train, n_val = int(n * 0.80), int(n * 0.10)
test_ds     = subset.select(range(n_train + n_val, n))
test_sample = test_ds.select(range(50))
print(f"Test split: {len(test_ds)} | Using 50 samples for eval")

# ── Load tokenizer ────────────────────────────────────────────────────────────
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, padding_side="left"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Load model + adapter ──────────────────────────────────────────────────────
print("Loading base model (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True,
)

#  Fix 1: re-enable KV cache (was disabled for training)
base_model.config.use_cache = True

#  Fix 2: silence the temperature/top_p/top_k warnings by overriding
#    the generation_config that ships with Qwen
base_model.generation_config = GenerationConfig(
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

print(f"Loading adapter from: {ADAPTER_DIR}")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
model.eval()
print("✓ Model + adapter ready")

Loading test split...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

Test split: 500 | Using 50 samples for eval
Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading adapter from: /content/drive/MyDrive/medical-reasoning-llm/results/final_adapter
✓ Model + adapter ready


In [ ]:
import re
import time
from rouge_score import rouge_scorer as rouge_lib

scorer = rouge_lib.RougeScorer(["rougeL"], use_stemmer=False)

# ── Parser helpers ────────────────────────────────────────────────────────────
def extract_answer(text):
    """Pull the text after 'Final Answer:' if present."""
    if "Final Answer:" in text:
        return text.split("Final Answer:", 1)[-1].strip()
    return text.strip().split("\n\n")[-1].strip()

def extract_reasoning(text):
    """Pull the CoT chain between 'Let me reason' and 'Final Answer:'."""
    if "Let me reason" in text and "Final Answer:" in text:
        return text.split("Let me reason", 1)[-1].split("Final Answer:", 1)[0].strip()
    return text.strip()

# ── Keyword-overlap metric (replaces broken exact-match) ─────────────────────
def keyword_match(pred, gold, threshold=0.30):
    """
    True if ≥30% of meaningful gold keywords appear in the prediction.
    This is what you should use instead of exact-match for free-form medical text.
    """
    stopwords = {
        "with","that","this","from","have","been","they","will","also",
        "some","more","most","when","than","into","each","over","such",
        "very","well","both","your","their","which","what","about","after",
    }
    gold_words = set(
        w.lower().strip(".,;:()[]") for w in gold.split()
        if len(w) >= 4 and w.lower() not in stopwords
    )
    if not gold_words:
        return False
    pred_lower = pred.lower()
    hits = sum(1 for w in gold_words if w in pred_lower)
    return (hits / len(gold_words)) >= threshold

# ── Inference loop ────────────────────────────────────────────────────────────
print("Running evaluation on 50 test examples...")
print("(max_new_tokens=200 + KV cache → ~5–8 min total)\n")

pred_answers, gold_answers = [], []
pred_chains,  gold_chains  = [], []
start = time.time()

model.eval()
for i, row in enumerate(test_sample):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": row["Question"]},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=200,       # ✅ was 512; 200 is plenty for eval
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_toks = out_ids[0][inputs["input_ids"].shape[1]:]
    output   = tokenizer.decode(new_toks, skip_special_tokens=True)

    pred_answers.append(extract_answer(output))
    gold_answers.append(row["Response"])
    pred_chains.append(extract_reasoning(output))
    gold_chains.append(row["Complex_CoT"])

    # Progress every 10 samples
    if (i + 1) % 10 == 0:
        elapsed = (time.time() - start) / 60
        print(f"  [{i+1:2d}/50]  {elapsed:.1f} min elapsed")

total_min = (time.time() - start) / 60

# ── Metrics ───────────────────────────────────────────────────────────────────
n = len(pred_answers)

# 1. Keyword-overlap accuracy (the real metric)
kw_correct = sum(keyword_match(p, g) for p, g in zip(pred_answers, gold_answers))
kw_acc     = kw_correct / n

# 2. ROUGE-L on reasoning chains
rouge_scores = [
    scorer.score(g, p)["rougeL"].fmeasure
    for p, g in zip(pred_chains, gold_chains)
]
avg_rouge = sum(rouge_scores) / len(rouge_scores)

# 3. Chain completeness (≥3 sentences of reasoning)
complete = sum(
    len(re.split(r"[.!?]\s+", c.strip())) >= 3
    for c in pred_chains
)
completeness = complete / n

# 4. Degenerate output rate (very short / empty answers)
empty = sum(len(p.strip()) < 30 for p in pred_answers)

print(f"\n  Finished in {total_min:.1f} min")
print("\n" + "=" * 55)
print("  EVALUATION RESULTS  (n=50)")
print("=" * 55)
print(f"  Keyword-overlap accuracy : {kw_acc:.4f}  ({kw_correct}/{n})")
print(f"  ROUGE-L (reasoning CoT)  : {avg_rouge:.4f}")
print(f"  Chain completeness       : {completeness:.4f}  ({complete}/{n})")
print(f"  Degenerate outputs       : {empty}/{n}")
print("=" * 55)
print(f"\n  ℹ  Exact-match = 0.0 always for free-form text — ignore it")
print(f"  ✅ Keyword accuracy is your real signal")

Running evaluation on 50 test examples...
(max_new_tokens=200 + KV cache → ~5–8 min total)



/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  [10/50]  6.3 min elapsed
  [20/50]  12.5 min elapsed
  [30/50]  18.7 min elapsed
  [40/50]  24.8 min elapsed
  [50/50]  31.0 min elapsed

  Finished in 31.0 min

  EVALUATION RESULTS  (n=50)
  Keyword-overlap accuracy : 0.0200  (1/50)
  ROUGE-L (reasoning CoT)  : 0.1768
  Chain completeness       : 1.0000  (50/50)
  Degenerate outputs       : 6/50

  ℹ  Exact-match = 0.0 always for free-form text — ignore it
  ✅ Keyword accuracy is your real signal


In [ ]:
print("Disabling adapter → running base model on same 50 questions...")
print("(Another ~5–8 min)\n")

model.disable_adapter_layers()
model.eval()

base_answers, base_chains = [], []
start = time.time()

for i, row in enumerate(test_sample):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": row["Question"]},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_toks = out_ids[0][inputs["input_ids"].shape[1]:]
    output   = tokenizer.decode(new_toks, skip_special_tokens=True)
    base_answers.append(extract_answer(output))
    base_chains.append(extract_reasoning(output))

    if (i + 1) % 10 == 0:
        print(f"  [{i+1:2d}/50]  {(time.time()-start)/60:.1f} min")

# Re-enable adapter for any future inference
model.enable_adapter_layers()
print("  ✓ Adapter re-enabled")

# Base metrics
base_kw    = sum(keyword_match(p, g) for p, g in zip(base_answers, gold_answers))
base_acc   = base_kw / n
base_rouge = sum(
    scorer.score(g, p)["rougeL"].fmeasure
    for p, g in zip(base_chains, gold_chains)
) / n
base_complete = sum(
    len(re.split(r"[.!?]\s+", c.strip())) >= 3 for c in base_chains
) / n

print("\n" + "=" * 60)
print("  BASE vs FINE-TUNED COMPARISON  (n=50)")
print("=" * 60)
print(f"  {'Metric':<30} {'Base':>8} {'Fine-tuned':>10} {'Δ':>8}")
print("  " + "─" * 56)
print(f"  {'Keyword Accuracy':<30} {base_acc:>8.4f} {kw_acc:>10.4f} {kw_acc - base_acc:>+8.4f}")
print(f"  {'ROUGE-L (CoT)':<30} {base_rouge:>8.4f} {avg_rouge:>10.4f} {avg_rouge - base_rouge:>+8.4f}")
print(f"  {'Chain Completeness':<30} {base_complete:>8.4f} {completeness:>10.4f} {completeness - base_complete:>+8.4f}")
print("=" * 60)

Disabling adapter → running base model on same 50 questions...
(Another ~5–8 min)

  [10/50]  3.8 min
  [20/50]  7.5 min
  [30/50]  11.2 min
  [40/50]  14.8 min
  [50/50]  18.7 min
  ✓ Adapter re-enabled

  BASE vs FINE-TUNED COMPARISON  (n=50)
  Metric                             Base Fine-tuned        Δ
  ────────────────────────────────────────────────────────
  Keyword Accuracy                 0.0200     0.0200  +0.0000
  ROUGE-L (CoT)                    0.1411     0.1768  +0.0356
  Chain Completeness               0.9800     1.0000  +0.0200


##################################################################

In [ ]:
# ── Proper evaluation: keyword-based diagnosis accuracy ──────────────────────
import re
from rouge_score import rouge_scorer as rouge_lib

scorer = rouge_lib.RougeScorer(["rougeL"], use_stemmer=False)

def extract_answer(text):
    if "Final Answer:" in text:
        return text.split("Final Answer:", 1)[-1].strip()
    return text.strip().split("\n\n")[-1].strip()

def extract_reasoning(text):
    if "Let me reason" in text and "Final Answer:" in text:
        return text.split("Let me reason", 1)[-1].split("Final Answer:", 1)[0].strip()
    return text.strip()

def keyword_match(pred, gold):
    """Check if key medical terms from gold appear in prediction."""
    # Extract meaningful words (4+ chars, not stopwords)
    stopwords = {"with","that","this","from","have","been","they",
                 "will","also","some","more","most","when","than",
                 "into","each","over","such","very","well","both"}
    gold_words = set(
        w.lower().strip(".,;:()") for w in gold.split()
        if len(w) >= 4 and w.lower() not in stopwords
    )
    pred_lower = pred.lower()
    if not gold_words:
        return False
    # Match if >30% of gold keywords appear in prediction
    hits = sum(1 for w in gold_words if w in pred_lower)
    return (hits / len(gold_words)) >= 0.30

print("Running evaluation on 50 test examples...")
print("(Using keyword overlap accuracy, not broken exact-match)\n")

pred_answers, gold_answers = [], []
pred_chains, gold_chains   = [], []

model.eval()
for row in test_sample:   # test_sample defined in Cell 13
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": row["Question"]},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_toks = out_ids[0][inputs["input_ids"].shape[1]:]
    output   = tokenizer.decode(new_toks, skip_special_tokens=True)

    pred_answers.append(extract_answer(output))
    gold_answers.append(row["Response"])
    pred_chains.append(extract_reasoning(output))
    gold_chains.append(row["Complex_CoT"])

# ── Metrics ───────────────────────────────────────────────────────────────────
n = len(pred_answers)

# 1. Keyword-overlap accuracy
kw_correct = sum(keyword_match(p, g) for p, g in zip(pred_answers, gold_answers))
kw_acc     = kw_correct / n

# 2. ROUGE-L on reasoning chains
rouge_scores = [
    scorer.score(g, p)["rougeL"].fmeasure
    for p, g in zip(pred_chains, gold_chains)
]
avg_rouge = sum(rouge_scores) / len(rouge_scores)

# 3. Chain completeness (has >= 3 sentences of reasoning)
complete = sum(
    len(re.split(r"[.!?]\s+", c.strip())) >= 3
    for c in pred_chains
)
completeness = complete / n

# 4. Empty/degenerate output rate
empty = sum(len(p.strip()) < 30 for p in pred_answers)

print("=" * 55)
print("  EVALUATION RESULTS  (n=50)")
print("=" * 55)
print(f"  Keyword-overlap accuracy : {kw_acc:.4f}  ({kw_correct}/{n})")
print(f"  ROUGE-L (reasoning CoT)  : {avg_rouge:.4f}")
print(f"  Chain completeness       : {completeness:.4f}  ({complete}/{n})")
print(f"  Degenerate outputs       : {empty}/{n}")
print("=" * 55)
print(f"\n  Old broken metric (exact-match): 0.0000 ← ignore")
print(f"  Real accuracy (keyword match)  : {kw_acc:.4f} ← use this")

Running evaluation on 50 test examples...
(Using keyword overlap accuracy, not broken exact-match)



/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  EVALUATION RESULTS  (n=50)
  Keyword-overlap accuracy : 0.7400  (37/50)
  ROUGE-L (reasoning CoT)  : 0.2014
  Chain completeness       : 1.0000  (50/50)
  Degenerate outputs       : 1/50

  Old broken metric (exact-match): 0.0000 ← ignore
  Real accuracy (keyword match)  : 0.7400 ← use this


In [ ]:
# ── Base model comparison (same 50 questions, no adapter) ────────────────────
from peft import PeftModel

print("Loading base model (no adapter) for comparison...")

# Temporarily disable adapter
model.disable_adapter_layers()
model.eval()

base_answers, base_chains = [], []

for row in test_sample:
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": row["Question"]},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_toks = out_ids[0][inputs["input_ids"].shape[1]:]
    output   = tokenizer.decode(new_toks, skip_special_tokens=True)
    base_answers.append(extract_answer(output))
    base_chains.append(extract_reasoning(output))

# Re-enable adapter
model.enable_adapter_layers()

# Base metrics
base_kw  = sum(keyword_match(p, g) for p, g in zip(base_answers, gold_answers))
base_acc = base_kw / n
base_rouge = sum(
    scorer.score(g, p)["rougeL"].fmeasure
    for p, g in zip(base_chains, gold_chains)
) / n

print("\n" + "=" * 55)
print("  BASE vs FINE-TUNED COMPARISON")
print("=" * 55)
print(f"  {'Metric':<30} {'Base':>8} {'FT':>8} {'Δ':>8}")
print("  " + "─" * 51)
print(f"  {'Keyword Accuracy':<30} {base_acc:>8.4f} {kw_acc:>8.4f} {kw_acc-base_acc:>+8.4f}")
print(f"  {'ROUGE-L (CoT)':<30} {base_rouge:>8.4f} {avg_rouge:>8.4f} {avg_rouge-base_rouge:>+8.4f}")
print("=" * 55)

Loading base model (no adapter) for comparison...

  BASE vs FINE-TUNED COMPARISON
  Metric                             Base       FT        Δ
  ───────────────────────────────────────────────────
  Keyword Accuracy                 0.1000   0.7400  +0.6400
  ROUGE-L (CoT)                    0.1675   0.2014  +0.0340
